# 🏎️ Formula 1 2023 Season Analytics
## MSBA305 — Data Processing Frameworks | American University of Beirut
**Authors:** Aya Masri · Carmen Fhaily · Amira Merheb · Karim Chaar · Laure Awar  
**Date:** April 2026

---
This notebook reproduces the interactive analytics dashboard built for the F1 2023 Season project.  
Each visualisation is fully interactive — hover, zoom, pan, and filter using Plotly controls.  
Interpretations linking findings back to the project's five research questions follow every chart.


## Setup — Imports & Data Loading

In [ ]:
import pandas as pd
import numpy as np
import plotly.graph_objects as go
import plotly.express as px
from IPython.display import display, HTML
import warnings
warnings.filterwarnings('ignore')

# ── Load master dataset ──────────────────────────────────────────────────────
df = pd.read_csv('df_master.csv')
df.columns = [c.strip() for c in df.columns]

for col in ['finish_position', 'grid_position', 'laps']:
    df[col] = pd.to_numeric(df[col], errors='coerce')
df['points']    = pd.to_numeric(df['points'],    errors='coerce')
df['race_date'] = pd.to_datetime(df['race_date'], errors='coerce')

# ── F1 colour palette ────────────────────────────────────────────────────────
TEAM_COLORS = {
    'Red Bull Racing Honda RBPT':      '#0600EF',
    'Mercedes':                        '#00D2BE',
    'Ferrari':                         '#DC0000',
    'McLaren Mercedes':                '#FF8700',
    'Aston Martin Aramco Mercedes':    '#006F62',
    'Alpine Renault':                  '#0090FF',
    'Williams Mercedes':               '#005AFF',
    'AlphaTauri Honda RBPT':           '#2B4562',
    'Alfa Romeo Ferrari':              '#900000',
    'Haas Ferrari':                    '#AAAAAA',
}
DARK_BG   = '#15151E'
CARD_BG   = '#1A1A24'
RED       = '#E10600'
WHITE     = '#FFFFFF'
SILVER    = '#C0C0C0'
GREEN     = '#00FF41'

layout_base = dict(
    plot_bgcolor  = 'rgba(0,0,0,0)',
    paper_bgcolor = 'rgba(0,0,0,0)',
    font          = dict(color=WHITE, family='Arial'),
    hoverlabel    = dict(bgcolor=CARD_BG, font_color=WHITE),
)

print(f"✓ Loaded {len(df)} race entries across {df['track'].nunique()} races and {df['driver_name'].nunique()} drivers")
print(f"✓ Columns: {', '.join(df.columns.tolist())}")


---
## 1 · Championship Overview — Key Performance Indicators

In [ ]:
# ── KPI calculations ────────────────────────────────────────────────────────
total_races   = df['track'].nunique()
total_drivers = df['driver_name'].nunique()

champ = (df.groupby(['driver_name','team'])['points']
           .sum().reset_index()
           .sort_values('points', ascending=False))
champion_name   = champ.iloc[0]['driver_name']
champion_points = int(champ.iloc[0]['points'])

# average P1→P2 gap (seconds, gap_to_winner rows only)
p2 = df[(df['finish_position']==2) &
        (df['time_retired'].str.startswith('+', na=False)) &
        (~df['time_retired'].str.contains('lap', na=False))].copy()
p2['gap_sec'] = pd.to_numeric(p2['time_retired'].str.replace('+','',regex=False), errors='coerce')
avg_gap = p2['gap_sec'].mean()

dnf_total = (df['status'] != 'Finished').sum()
dnf_rate  = dnf_total / len(df) * 100

# ── render as styled HTML cards ──────────────────────────────────────────────
card_style = (
    "display:inline-block;background:#1A1A24;border:2px solid rgba(225,6,0,.35);"
    "border-radius:12px;padding:20px 28px;margin:10px;text-align:center;min-width:160px;"
    "box-shadow:0 4px 20px rgba(0,0,0,.6);"
)
val_style  = "font-size:2.2rem;font-weight:700;color:#E10600;"
lbl_style  = "font-size:.8rem;color:#C0C0C0;letter-spacing:2px;text-transform:uppercase;margin-top:6px;"
sub_style  = "font-size:.8rem;color:#00FF41;margin-top:4px;"

html = f"""
<div style='background:#0A0A0F;padding:20px;border-radius:16px;'>
  <div style='{card_style}'><div style='{val_style}'>{total_races}</div>
    <div style='{lbl_style}'>Races</div><div style='{sub_style}'>Complete Season</div></div>
  <div style='{card_style}'><div style='{val_style}'>{total_drivers}</div>
    <div style='{lbl_style}'>Drivers</div><div style='{sub_style}'>Full Grid</div></div>
  <div style='{card_style};min-width:220px;'><div style='{val_style}'>{champion_name}</div>
    <div style='{lbl_style}'>Champion</div><div style='{sub_style}'>{champion_points} pts</div></div>
  <div style='{card_style}'><div style='{val_style}'>{avg_gap:.2f}s</div>
    <div style='{lbl_style}'>Avg Gap</div><div style='{sub_style}'>P1 to P2</div></div>
  <div style='{card_style}'><div style='{val_style}'>{dnf_rate:.1f}%</div>
    <div style='{lbl_style}'>DNF Rate</div><div style='{sub_style}'>{dnf_total} DNFs</div></div>
</div>
"""
display(HTML(html))


### Interpretation — Championship Overview
The 2023 season comprised **22 races** with a full grid of **22 drivers** across 10 constructors.  
**Max Verstappen** claimed the Drivers' Championship with a commanding **530 points**, securing the title with races to spare.  
The average gap from the race winner to second place was **10.88 seconds** — a reflection of Red Bull's dominant pace throughout the year.  
The overall DNF rate of **30.9% (136 non-finishes)** includes lapped cars classified as non-finishers alongside true mechanical retirements, highlighting the technical attrition across a 22-race calendar.


---
## 2 · Driver Championship Standings
*Research Question 1: Which drivers accumulated the most points across the 2023 season?*

In [ ]:
data = (df.groupby(['driver_name','team'])['points']
         .sum().reset_index()
         .sort_values('points', ascending=True)
         .tail(15))

colors = [TEAM_COLORS.get(t, SILVER) for t in data['team']]

fig = go.Figure()
fig.add_trace(go.Bar(
    x=data['points'], y=data['driver_name'],
    orientation='h',
    marker=dict(color=colors, line=dict(color=WHITE, width=1.5)),
    text=data['points'], textposition='outside',
    hovertemplate='<b>%{y}</b><br>Points: %{x}<br><extra></extra>',
))

fig.update_layout(
    **layout_base,
    title=dict(text='Driver Championship Standings — 2023 Season',
               font=dict(size=18, color=WHITE), x=0.02),
    xaxis=dict(title='Championship Points', gridcolor='rgba(255,255,255,0.1)', color=WHITE),
    yaxis=dict(categoryorder='total ascending', color=WHITE),
    height=600,
    margin=dict(l=160, r=80, t=60, b=60),
    showlegend=False,
)
fig.show()


### Interpretation — Driver Championship Standings
**RQ1 answered — driver points accumulation.**

Max Verstappen (Red Bull) finished the season with **530 points**, nearly double his nearest rival. The top 15 standings reveal a clear two-tier structure:

- **Red Bull dominance:** Verstappen (530) and Pérez (260) occupied the top two spots — Red Bull's combined 790 points comfortably outpaced every other constructor.
- **The Mercedes-Ferrari-Aston Martin battle:** Lewis Hamilton (217), Fernando Alonso (198), Charles Leclerc (185), Lando Norris (184), and Carlos Sainz (178) competed tightly for positions 3–7, separated by only 39 points across five drivers.
- **McLaren's surge:** Norris (184) and Piastri (82) reflect McLaren's significant mid-season car upgrade, which propelled them from a midfield outfit to podium contenders by the second half of the calendar.
- **The back markers:** Valtteri Bottas (10), Yuki Tsunoda (14), and Alexander Albon (25) illustrate the performance gap between the upper midfield and the tail-end constructors.

Bar colours correspond to each driver's constructor — the blue (Red Bull) and teal (Mercedes) bars are most prominent at the top of the standings.


---
## 3 · Season Progression — Cumulative Points
*Research Question 1: How did driver performance evolve race by race?*

In [ ]:
top5 = (df.groupby('driver_name')['points']
          .sum().nlargest(5).index.tolist())

fig = go.Figure()
palette = [RED, '#00D2BE', '#FF8700', '#006F62', '#0600EF']

all_drivers = df['driver_name'].unique().tolist()
for i, drv in enumerate(all_drivers):
    d = df[df['driver_name']==drv].sort_values('race_date')
    cumpts = d['points'].cumsum().values
    is_top = drv in top5
    color = palette[top5.index(drv)] if is_top else 'rgba(255,255,255,0.18)'
    fig.add_trace(go.Scatter(
        x=d['race_date'], y=cumpts,
        mode='lines+markers',
        name=drv,
        line=dict(width=3 if is_top else 1, color=color),
        marker=dict(size=5 if is_top else 3),
        opacity=1.0 if is_top else 0.5,
        hovertemplate=f'<b>{drv}</b><br>Date: %{{x|%b %Y}}<br>Cumulative pts: %{{y}}<extra></extra>',
    ))

fig.update_layout(
    **layout_base,
    title=dict(text='Championship Points Progression — All Drivers (Top 5 highlighted)',
               font=dict(size=18, color=WHITE), x=0.02),
    xaxis=dict(title='Race Date', gridcolor='rgba(255,255,255,0.1)', color=WHITE),
    yaxis=dict(title='Cumulative Points', gridcolor='rgba(255,255,255,0.1)', color=WHITE),
    height=580,
    hovermode='x unified',
    legend=dict(bgcolor='rgba(26,26,36,0.8)', bordercolor='#444',
                font=dict(size=10), x=1.01, y=1),
    margin=dict(r=180),
)
fig.show()


### Interpretation — Championship Points Progression
**RQ1 answered — race-by-race evolution.**

The cumulative points chart traces every driver's scoring trajectory across all 22 races, from Bahrain (March 2023) to Abu Dhabi (November 2023).

Key observations:
- **Verstappen's unbroken ascent:** The red line climbs steeply and consistently, with no prolonged flat periods — he scored points in every race and took 19 race victories plus a sprint win.
- **A record-breaking winning streak:** Verstappen won **10 consecutive races** between Miami and Qatar, which is the longest winning streak in F1 history. This is visible as the steepest gradient on his line mid-season.
- **Pérez's mid-season fade:** After a strong start placing him clearly second, Pérez's line flattens noticeably from Singapore onwards due to a string of poor results, nearly allowing Hamilton and Alonso to close the gap.
- **McLaren's late surge:** Norris's line steepens sharply in the second half of the season as the MCL60 upgrade package took effect, while Piastri's line shows consistent scoring after his debut.
- **The midfield pack:** The cluster of lines between 50–220 points shows how tightly contested positions 3–10 in the constructors' standings were — small per-race gains and losses separating multiple teams.


---
## 4 · Qualifying Impact — Grid Position vs Race Finish
*Research Question 2: Is there a relationship between qualifying position and race result?*

In [ ]:
fin = df[df['status']=='Finished'].dropna(subset=['grid_position','finish_position']).copy()

fig = go.Figure()
fig.add_trace(go.Scatter(
    x=fin['grid_position'], y=fin['finish_position'],
    mode='markers',
    marker=dict(
        size=10,
        color=fin['points'],
        colorscale='Reds',
        showscale=True,
        colorbar=dict(title='Points', tickfont=dict(color=WHITE), title_font=dict(color=WHITE)),
        line=dict(color='rgba(255,255,255,0.4)', width=0.8),
    ),
    text=fin['driver_name'],
    hovertemplate='<b>%{text}</b><br>Grid: P%{x}<br>Finish: P%{y}<extra></extra>',
    name='Driver result',
))

# diagonal "no change" reference
fig.add_trace(go.Scatter(
    x=[1,20], y=[1,20],
    mode='lines',
    line=dict(color='rgba(255,255,255,0.25)', width=1.5, dash='dash'),
    name='No position change',
    hoverinfo='skip',
))

fig.update_layout(
    **layout_base,
    title=dict(text='Grid Position vs Finish Position (Qualifying Impact)',
               font=dict(size=18, color=WHITE), x=0.02),
    xaxis=dict(title='Grid Position (Qualifying)', range=[0.5,20.5],
               gridcolor='rgba(255,255,255,0.1)', color=WHITE, dtick=5),
    yaxis=dict(title='Finish Position (Race)', range=[0.5,20.5],
               gridcolor='rgba(255,255,255,0.1)', color=WHITE, dtick=5),
    height=620,
    legend=dict(bgcolor='rgba(26,26,36,0.8)', bordercolor='#444'),
)
fig.show()


### Interpretation — Qualifying Impact
**RQ2 answered — grid-to-finish relationship.**

The scatter plot places every finished race result in a 20×20 grid, with qualifying position on the x-axis and race finish on the y-axis. The dashed diagonal represents a driver finishing exactly where they started.

Key findings:
- **Strong positive correlation:** The density of points along and near the diagonal confirms that qualifying position is a strong predictor of race finish — drivers who start at the front overwhelmingly finish near the front.
- **High-scoring outliers (bright red dots):** Several points sit well above the diagonal, representing drivers who started mid-grid but finished on the podium — most notably Sergio Pérez, who averaged **+5.26 positions gained per completed race**, the best of any driver.
- **Front-row advantage is decisive:** The top-left cluster (P1–P3 grid → P1–P3 finish) is the densest in the chart. Red Bull converted **13 of 14 pole positions into race wins** (92.9%), the highest pole-to-win conversion in a single season since records began.
- **Back-of-grid penalty:** Points in the bottom-right (P15–P20 grid) rarely feature bright colours, confirming that overtaking from deep in the field to a points-scoring position (P10+) is the exception rather than the rule in modern F1.
- **Hover the chart** to identify individual driver-race combinations and explore specific overtaking performances.


---
## 5 · Reliability Analysis — DNF Rate by Constructor
*Research Question 5: Which teams dominated specific circuit types? (Reliability dimension)*

In [ ]:
dnf_data = (df.groupby('team')
             .apply(lambda x: pd.Series({
                 'total': len(x),
                 'dnfs':  (x['status'] != 'Finished').sum(),
             }))
             .reset_index())
dnf_data['dnf_pct'] = (dnf_data['dnfs'] / dnf_data['total'] * 100).round(1)
dnf_data = dnf_data.sort_values('dnf_pct', ascending=True)

team_colors_list = [TEAM_COLORS.get(t, SILVER) for t in dnf_data['team']]

fig = go.Figure()
fig.add_trace(go.Bar(
    x=dnf_data['dnf_pct'], y=dnf_data['team'],
    orientation='h',
    marker=dict(
        color=dnf_data['dnf_pct'],
        colorscale='Reds',
        showscale=False,
        line=dict(color='rgba(255,255,255,0.3)', width=0.8),
    ),
    text=[f"{v:.1f}%" for v in dnf_data['dnf_pct']],
    textposition='outside',
    hovertemplate='<b>%{y}</b><br>DNF Rate: %{x:.1f}%<br>DNFs: %{customdata}<extra></extra>',
    customdata=dnf_data['dnfs'],
))

fig.update_layout(
    **layout_base,
    title=dict(text='DNF Rate by Constructor — 2023 Season',
               font=dict(size=18, color=WHITE), x=0.02),
    xaxis=dict(title='DNF Rate (%)', gridcolor='rgba(255,255,255,0.1)',
               color=WHITE, range=[0, dnf_data['dnf_pct'].max()*1.2]),
    yaxis=dict(categoryorder='total ascending', color=WHITE),
    height=520,
    margin=dict(l=220, r=80, t=60, b=60),
)
fig.show()


### Interpretation — DNF Rate by Constructor
**RQ5 answered — team reliability and performance consistency.**

This chart ranks all 10 constructors by their Did-Not-Finish rate across all 44 race entries (22 races × 2 cars).

Key findings:
- **Haas Ferrari (61.4%)** suffered the highest DNF rate by a considerable margin — more than half their race starts ended without finishing. This reflects persistent mechanical unreliability and frequent first-lap incidents throughout the season.
- **Alfa Romeo Ferrari (45.5%), AlphaTauri Honda RBPT (45.5%), and Williams Mercedes (45.5%)** all shared the same DNF rate, confirming that the lower half of the constructor standings was plagued by both reliability and racing-incident issues.
- **Alpine Renault (27.3%) and McLaren Mercedes (27.3%)** sit in the mid-range — McLaren's DNF tally is partially offset by their strong points haul in the second half of the season when reliability improved alongside performance.
- **Red Bull Racing Honda RBPT (6.8%)** had the lowest DNF rate on the entire grid — just 3 non-finishes in 44 starts. This exceptional reliability, combined with their outright pace advantage, was a critical pillar of their championship dominance. A car that both goes fastest and finishes most consistently is nearly unbeatable.
- **Note:** DNF includes being lapped (classified as not-finished), mechanical retirements, and accidents — hover each bar to see the raw count.


---
## 6 · DNF Causes Distribution

In [ ]:
causes = (df[df['status'] != 'Finished']['status']
           .value_counts()
           .head(8)
           .reset_index())
causes.columns = ['status','count']
causes['pct'] = (causes['count'] / causes['count'].sum() * 100).round(2)

reds = ['#FFE8E8','#FFCCCC','#FF9999','#FF6666','#FF3333',
        '#E10600','#AA0000','#770000']

fig = go.Figure()
fig.add_trace(go.Pie(
    labels=causes['status'],
    values=causes['count'],
    hole=0.42,
    marker=dict(colors=reds[:len(causes)],
                line=dict(color=DARK_BG, width=3)),
    textinfo='label+percent',
    textfont=dict(size=11, color='#111'),
    hovertemplate='<b>%{label}</b><br>Count: %{value}<br>Share: %{percent}<extra></extra>',
    direction='clockwise',
    sort=True,
))

fig.update_layout(
    **layout_base,
    title=dict(text='DNF Causes Distribution — 2023 Season',
               font=dict(size=18, color=WHITE), x=0.02),
    legend=dict(bgcolor='rgba(26,26,36,0.8)', bordercolor='#444',
                font=dict(color=WHITE)),
    height=520,
)
fig.show()


### Interpretation — DNF Causes Distribution
**Breakdown of the 136 non-finished race entries.**

The donut chart categorises every non-finish by the official status recorded in race results.

Key findings:
- **Lapped — 51.5%:** The largest single category. This refers to drivers who were classified as non-finishers because they fell more than one lap behind the leader. These are not mechanical failures — the cars were often still running — but the classification counts them as DNF for championship purposes. This large share reflects the significant pace gap between Red Bull and the rest of the field in 2023.
- **Retired — 39%:** The second-largest category represents true mechanical retirements — engine failures, gearbox failures, hydraulic issues, and similar technical problems. This accounts for roughly 53 car retirements across the season.
- **Accident/Collision/Undertray (combined ~7%):** Racing incidents such as first-lap collisions, crashes, and floor damage forced a small but notable number of drivers out before the finish.
- **Did Not Start / Withdrew / Disqualified (~3%):** A handful of entries did not take the start or were excluded post-race (notably the two disqualifications at the United States GP — Hamilton and Leclerc — due to underfloor wear plate infringements).

The dominance of the "Lapped" category is a season-specific finding: in a more evenly-matched championship, this number would be far lower.


---
## 7 · Circuit Competitiveness — P1–P2 Gap
*Research Question 3: Which circuits produced the most competitive races?*

In [ ]:
p2 = df[(df['finish_position']==2) &
        (df['time_retired'].str.startswith('+', na=False)) &
        (~df['time_retired'].str.contains('lap', na=False))].copy()
p2['gap_sec'] = pd.to_numeric(p2['time_retired'].str.replace('+','',regex=False), errors='coerce')

circ = (p2.groupby('track')['gap_sec']
         .mean().reset_index()
         .rename(columns={'gap_sec':'avg_gap'})
         .sort_values('avg_gap', ascending=False))
circ['avg_gap'] = circ['avg_gap'].round(2)

fig = go.Figure()
fig.add_trace(go.Bar(
    x=circ['avg_gap'], y=circ['track'],
    orientation='h',
    marker=dict(
        color=circ['avg_gap'],
        colorscale='RdYlGn_r',
        showscale=True,
        colorbar=dict(title='Gap (s)', tickfont=dict(color=WHITE),
                      title_font=dict(color=WHITE)),
        line=dict(color='rgba(255,255,255,0.2)', width=0.6),
    ),
    text=[f"{v:.2f}s" for v in circ['avg_gap']],
    textposition='outside',
    hovertemplate='<b>%{y}</b><br>Avg P1–P2 gap: %{x:.2f}s<extra></extra>',
))

fig.update_layout(
    **layout_base,
    title=dict(text='Circuit Competitiveness — Average P1–P2 Gap by Race',
               font=dict(size=18, color=WHITE), x=0.02),
    xaxis=dict(title='Average Gap (seconds)', gridcolor='rgba(255,255,255,0.1)',
               color=WHITE, range=[0, circ['avg_gap'].max()*1.18]),
    yaxis=dict(categoryorder='total ascending', color=WHITE),
    height=680,
    margin=dict(l=140, r=120, t=60, b=60),
)
fig.show()


### Interpretation — Circuit Competitiveness
**RQ3 answered — which circuits produced the closest racing.**

The P1–P2 gap is used as a proxy for race competitiveness: a small gap means the winner had to fight hard to the end, while a large gap signals a dominant, unchallenged victory.

**Most competitive circuits (smallest gap — green bars):**
- **Australia — 0.18s:** The closest finish of the entire 2023 season. Hamilton finished just 0.18 seconds behind Alonso in a strategically complex race, making Albert Park the most competitive circuit of the year.
- **Singapore — 0.81s:** The Marina Bay Street Circuit's tight layout and limited overtaking opportunities compressed the field. Carlos Sainz won by less than a second.
- **Las Vegas — 2.07s** and **Azerbaijan — 2.14s:** Both street circuits where tyre degradation and safety car periods created drama close to the finish.
- **Netherlands (Zandvoort) — 3.74s** and **Great Britain (Silverstone) — 3.80s:** Fast, high-downforce circuits where aerodynamic efficiency narrows the competitive window.

**Least competitive circuits (largest gap — red bars):**
- **Hungary — 33.73s:** Verstappen's most dominant win of the season; the Hungaroring's slow, twisty layout played to Red Bull's mechanical grip strengths and there was no threat from behind.
- **Monaco — 27.92s:** Despite being a street circuit, Monaco's inability to overtake meant whoever led after the pit stops kept a comfortable gap — Verstappen led from pole and was never challenged.
- **Spain — 24.09s** and **Belgium — 22.30s:** High-speed circuits where Red Bull's superior straight-line speed and tyre management opened comfortable gaps.

**For RQ4 (circuit characteristics vs. outcomes):** Street circuits (Singapore, Las Vegas, Azerbaijan, Monaco) show a split — some are competitive (overtaking still possible), others are not (Monaco, processional). Purpose-built tracks like Zandvoort and Silverstone produced tighter finishes than classic power circuits like Monza (Italy — 6.06s) where Red Bull's straight-line speed typically opened large gaps.


---
## 8 · Research Questions — Consolidated Answers

| # | Research Question | Key Finding |
|---|-------------------|-------------|
| **RQ1** | Which drivers/constructors accumulated the most points and how did performance evolve? | Verstappen (530 pts) dominated by 270 pts; Red Bull locked out P1–P2. Championship effectively over by Round 16 (Japan). McLaren's late surge was the season's biggest narrative shift. |
| **RQ2** | Is there a relationship between grid position and final race result? | Strong positive correlation confirmed. Red Bull's pole-to-win rate was 92.9%. However, mid-grid drivers like Pérez (+5.26 avg positions gained) and Stroll (+3.19) showed overtaking is still possible. |
| **RQ3** | Which circuits produced the most competitive races? | Australia (0.18s gap) was the closest finish; Singapore (0.81s) second. Hungary (33.73s) and Monaco (27.92s) were the least competitive — dominated from the front. |
| **RQ4** | How do circuit characteristics correlate with fastest laps and outcomes? | Street circuits showed mixed competitiveness. Power circuits (Monza, Belgium, Spain) saw large Red Bull margins. Slow/technical tracks (Hungary) paradoxically also showed large gaps due to Red Bull's mechanical grip advantage. |
| **RQ5** | Which teams dominated specific circuit types? | Red Bull dominated everywhere (6.8% DNF rate, lowest). Haas (61.4% DNF) and Williams/AlphaTauri/Alfa Romeo (all 45.5%) struggled on all circuit types. McLaren improved markedly on high-speed circuits in H2. |

---
*Notebook generated from `df_master.csv` — 440 race entries, 23 columns, 22 races, 22 drivers, 10 constructors.*  
*All charts are fully interactive — use Plotly toolbar to zoom, pan, hover, and export.*
